<a href="https://colab.research.google.com/github/Cosmosmk2/MNIST-Image-Classifier/blob/main/Mnist_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

In [4]:
mnisttrain = datasets.MNIST(root = './root', train = True, download = True, transform = train_transform)
mnisttest = datasets.MNIST(root = './root', train = False, download = True, transform = test_transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 332kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.14MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.05MB/s]


In [5]:
trainloader = DataLoader(mnisttrain, batch_size = 64, shuffle = True)
testloader = DataLoader(mnisttest, batch_size = 64, shuffle = False)

In [6]:
class Block(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels = in_channels,
                      out_channels = out_channels,
                      kernel_size = 3,
                      stride = 1,
                      padding = 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            )

    def forward(self, x):
        return self.layers(x)

class CNNmodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            Block(1, 32),
            Block(32, 64),
            Block(64, 128),
            Block(128, 256),
            Block(256, 512),
            nn.AdaptiveAvgPool2d((3, 3))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 3 * 3, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10)
        )
    def forward(self, x):
        x = self.layers(x)
        x = self.fc(x)

        return x

In [7]:
model = CNNmodel().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.0005)

In [8]:
def Training(loader, model, loss_function, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch_index, (entry, label) in enumerate(loader):
        entry = entry.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(entry)
        loss = loss_function(output, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = output.max(1)
        total += label.size(0)
        correct += predicted.eq(label).sum().item()

        if batch_index % 100 == 0:
            avg_loss = running_loss / (batch_index + 1)
            accuracy = 100 * correct / total
            print(f'Batch: {batch_index} | loss: {avg_loss:.4f} | accuracy: {accuracy:.2f}%')


def Evaluate(loader, model, device):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for batch_index, (entry, label) in enumerate(loader):
            entry = entry.to(device)
            label = label.to(device)

            output = model(entry)
            _, predicted = output.max(1)
            total += label.size(0)
            correct += predicted.eq(label).sum().item()

            if batch_index % 100 == 0:
                accuracy = 100 * correct / total
                print(f'batch: {batch_index} | accuracy: {accuracy:.2f}%')

        final_accuracy = 100 * correct / total
        print(f'final accuracy: {final_accuracy:.2f}%')

In [10]:
epochs = 100
for epoch in range(epochs):
    print(f'epoch: {epoch}')
    Training(loader = trainloader, model = model, loss_function = loss_function, optimizer = optimizer, device = device)
    Evaluate(loader = testloader, model = model, device = device)

epoch: 0
Batch: 0 | loss: 2.3217 | accuracy: 9.38%
Batch: 100 | loss: 1.0728 | accuracy: 63.01%
Batch: 200 | loss: 0.8162 | accuracy: 71.87%
Batch: 300 | loss: 0.6850 | accuracy: 76.47%
Batch: 400 | loss: 0.5981 | accuracy: 79.58%
Batch: 500 | loss: 0.5369 | accuracy: 81.78%
Batch: 600 | loss: 0.4905 | accuracy: 83.43%
Batch: 700 | loss: 0.4509 | accuracy: 84.83%
Batch: 800 | loss: 0.4184 | accuracy: 85.93%
Batch: 900 | loss: 0.3924 | accuracy: 86.82%
batch: 0 | accuracy: 93.75%
batch: 100 | accuracy: 94.74%
final accuracy: 95.42%
epoch: 1
Batch: 0 | loss: 0.2110 | accuracy: 93.75%
Batch: 100 | loss: 0.1727 | accuracy: 94.68%
Batch: 200 | loss: 0.1636 | accuracy: 94.80%
Batch: 300 | loss: 0.1650 | accuracy: 94.76%
Batch: 400 | loss: 0.1589 | accuracy: 94.95%
Batch: 500 | loss: 0.1552 | accuracy: 95.03%
Batch: 600 | loss: 0.1515 | accuracy: 95.20%
Batch: 700 | loss: 0.1491 | accuracy: 95.25%
Batch: 800 | loss: 0.1479 | accuracy: 95.32%
Batch: 900 | loss: 0.1446 | accuracy: 95.41%
batch:

KeyboardInterrupt: 